# Non-faithful embedding witness

This notebook plots a minimal Level 4 witness for a non-faithful embedding. The abstract simplicial complex has two 2-simplices, $[0,1,2]$ and $[0,1,3]$, sharing exactly the edge $[0,1]$.

A faithful embedding would make their geometric intersection equal to that shared edge. Here the second triangle lies inside the first, so the geometric intersection has positive area. Full Delaunay validation must reject this at Level 4 before any Level 5 empty-circumsphere reasoning is trusted.

## Delaunay caveat

Yes: this configuration is not a valid Delaunay triangulation. That is the point of the example. In this crate, `validate()` is cumulative: Level 5 is checked only after Levels 1-4 establish a valid embedded triangulation. A non-faithful embedding is therefore already invalid before the Delaunay property is meaningful.

This plot is useful when explaining why faithful embedding is a separate validation layer: geometric overlap outside shared faces is an invalid triangulation state, not a Delaunay repair target.

In [ ]:
"""Plot a minimal non-faithful embedding witness."""

from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Polygon

type Coordinate = tuple[float, float]
type Triangle = tuple[Coordinate, Coordinate, Coordinate]


def find_repo_root(start: Path) -> Path:
    """Return the repository root containing the project marker files."""
    for path in (start, *start.parents):
        if (path / "pyproject.toml").is_file() and (path / "Cargo.toml").is_file():
            return path
    msg = "Run this notebook from inside the delaunay repository."
    raise RuntimeError(msg)


ROOT = find_repo_root(Path.cwd().resolve())
FIGURE_DIR = ROOT / "target" / "notebooks" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_PATH = FIGURE_DIR / "nonfaithful_embedding.png"

first_triangle: Triangle = ((0.0, 0.0), (1.0, 0.0), (0.0, 1.0))
second_triangle: Triangle = ((0.0, 0.0), (1.0, 0.0), (0.25, 0.25))
shared_edge: tuple[Coordinate, Coordinate] = (first_triangle[0], first_triangle[1])


## What the plot shows

Topologically, the two triangles share only the black edge $[0,1]$. Geometrically, the orange triangle $[0,1,3]$ is contained in the blue triangle $[0,1,2]$. The hatched orange area is therefore an illegal intersection outside the shared face.

In the crate's terms, this is the pure-geometry witness behind `SimplexIntersectionFailure::IntersectionOutsideSharedFace` and the triangulation-level `TriangulationEmbeddingValidationError::SimplexIntersectionOutsideSharedFace`.

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 6.0), layout="constrained")

first_patch = Polygon(
    first_triangle,
    closed=True,
    facecolor="#6baed6",
    edgecolor="#08519c",
    linewidth=2.0,
    alpha=0.35,
    label="simplex [0, 1, 2]",
)
second_patch = Polygon(
    second_triangle,
    closed=True,
    facecolor="#fdae6b",
    edgecolor="#a63603",
    linewidth=2.0,
    alpha=0.7,
    hatch="///",
    label="simplex [0, 1, 3] (illegal overlap)",
)
ax.add_patch(first_patch)
ax.add_patch(second_patch)

edge_x = [shared_edge[0][0], shared_edge[1][0]]
edge_y = [shared_edge[0][1], shared_edge[1][1]]
ax.plot(edge_x, edge_y, color="black", linewidth=3.0, label="shared topological edge [0, 1]")

labels = {
    "0": first_triangle[0],
    "1": first_triangle[1],
    "2": first_triangle[2],
    "3": second_triangle[2],
}
for label, (x_coord, y_coord) in labels.items():
    ax.scatter([x_coord], [y_coord], color="black", s=36, zorder=4)
    ax.annotate(
        label,
        xy=(x_coord, y_coord),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=12,
        weight="bold",
    )

ax.annotate(
    "geometric intersection is area,\nnot just the shared edge",
    xy=(0.28, 0.12),
    xytext=(0.46, 0.44),
    arrowprops={"arrowstyle": "->", "color": "#a63603", "linewidth": 1.8},
    color="#7f2704",
    fontsize=11,
)
ax.set_title("A minimal non-faithful embedding")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-0.12, 1.12)
ax.set_ylim(-0.12, 1.12)
ax.grid(color="#d9d9d9", linewidth=0.8)
ax.legend(loc="upper right", frameon=True)

fig.savefig(FIGURE_PATH, dpi=200)
plt.show()


## Copyable summary

This is intentionally a Level 4 example, not a valid Delaunay triangulation. The abstract complex has two triangles, $[0,1,2]$ and $[0,1,3]$, whose intersection in the complex is the edge $[0,1]$. With coordinates $0=(0,0)$, $1=(1,0)$, $2=(0,1)$, and $3=(0.25,0.25)$, the second triangle lies inside the first. Thus $|[0,1,2]| \cap |[0,1,3]|$ contains a two-dimensional region even though $[0,1,2] \cap [0,1,3]$ is only one-dimensional. Full Delaunay validation must reject this before the Level 5 Delaunay property is evaluated, because the empty-circumsphere check presupposes a faithful embedded triangulation.

In [ ]:
if not FIGURE_PATH.is_file():
    msg = f"expected saved plot at {FIGURE_PATH}"
    raise FileNotFoundError(msg)